# Pelajaran 12 - Pengurangan Riwayat Obrolan dengan Agent Scratchpad

Notebook ini menunjukkan cara mengelola konteks dalam percakapan panjang menggunakan Microsoft Agent Framework. Saat percakapan berkembang, jumlah token meningkat — akhirnya melebihi jendela konteks model. Kami mengatasi ini dengan **pola rangkuman konteks** dan **agent scratchpad** untuk memori yang persisten.

## Apa yang Akan Anda Pelajari:
1. **Mengapa Manajemen Konteks Penting**: Memahami batas token dan jendela konteks
2. **Agent yang Memahami Konteks**: Membangun agent yang mengelola konteks percakapan mereka sendiri
3. **Pola Rangkuman Konteks**: Menggunakan alat untuk merangkum riwayat percakapan
4. **Agent Scratchpad**: Memori persisten yang bertahan setelah reduksi konteks

## Prasyarat:
- Pengaturan Azure OpenAI dengan variabel lingkungan yang dikonfigurasi
- Pemahaman konsep agent dasar dari pelajaran sebelumnya


## Pengaturan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Mengapa Manajemen Konteks Penting

Setiap LLM memiliki **jendela konteks** terbatas — jumlah maksimum token yang dapat diproses dalam satu permintaan. Seiring berjalannya percakapan multi-putar:

- **Jumlah token tumbuh secara linear** dengan setiap pesan pengguna dan balasan asisten.
- **Token prompt mendominasi biaya** karena seluruh riwayat dikirim ulang setiap putaran.
- Akhirnya percakapan **melebihi jendela konteks** dan model akan memotong atau menghasilkan kesalahan.

### Strategi untuk Mengelola Konteks

| Strategi | Cara Kerjanya | Pertukaran |
|---|---|---|
| **Pemotongan** | Menghapus pesan tertua | Kehilangan konteks awal |
| **Ringkasan** | Mengkondensasi pesan lama menjadi ringkasan | Beberapa detail hilang, tapi poin utama tetap |
| **Scratchpad / Memori Eksternal** | Menyimpan fakta kunci di luar percakapan | Membutuhkan panggilan alat, tapi bertahan dari pengurangan apa pun |

Dalam notebook ini kami menggabungkan **ringkasan** dengan **alat scratchpad** sehingga agen dapat mempertahankan kontinuitas meskipun riwayat percakapan dipadatkan.


## Membuat Agen yang Sadar Konteks


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Mensimulasikan Percakapan Panjang

Mari kita jalani percakapan multi-putar untuk melihat bagaimana konteks terakumulasi. Agen harus menyimpan detail kunci (preferensi, anggaran, tanggal perjalanan) di setiap putaran dan menunjukkan kesinambungan.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Perhatikan bagaimana agen mempertahankan konteks dari giliran sebelumnya — ia mengetahui tentang Jepang, sushi, kuil, fotografi, anggaran $3000, perjalanan solo, dan waktu April. Dalam percakapan singkat ini bekerja dengan baik, tetapi seiring percakapan berkembang, keseluruhan riwayat menjadi mahal untuk dikirim ulang.

Mari lanjutkan percakapan dengan lebih banyak giliran untuk melihat akumulasi konteks:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Pola Ringkasan Konteks

Seiring percakapan berkembang, kita dapat menggunakan **alat ringkasan** untuk mengkondensasikan konteks yang terkumpul menjadi format yang ringkas. Agen memanggil alat ini untuk merekam preferensi kunci sehingga meskipun pesan lama dihapus, informasi penting tetap terjaga.

Pola ini adalah blok bangunan untuk pengurangan riwayat yang lebih canggih:  
1. Agen mengidentifikasi fakta kunci dari percakapan  
2. Agen memanggil alat ringkasan untuk menyimpannya  
3. Pesan lama dapat dihapus dengan aman karena ringkasan menangkap hal-hal yang penting  

Di bawah ini kami mendefinisikan alat `summarize_preferences` yang dapat dipanggil agen untuk merekam ringkasan singkat dari apa yang telah dipelajarinya.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Ringkasan

Dalam pelajaran ini Anda telah belajar bagaimana mengelola konteks dalam percakapan agen yang berjalan lama menggunakan Microsoft Agent Framework:

### Konsep Utama
- **Jendela konteks bersifat terbatas** — setiap token dalam riwayat percakapan memerlukan biaya dan dihitung ke dalam batasan.
- **Alat ringkasan** memungkinkan agen merangkum konteks yang terkumpul menjadi ringkasan yang padat, mengurangi penggunaan token sambil mempertahankan informasi penting.
- **Scratchpad agen** menyediakan memori eksternal yang persisten yang bertahan meskipun percakapan dipangkas.

### Apa yang Anda Bangun
- Sebuah **agen yang sadar konteks** yang mempertahankan kontinuitas di seluruh percakapan multi-putar
- Sebuah **alat ringkasan** (`summarize_preferences`) yang mencatat detail utama pengguna dalam format yang ringkas
- Sebuah **percakapan multi-putar** yang menunjukkan retensi konteks dan penanganan perubahan

### Aplikasi Dunia Nyata
- **Bot Layanan Pelanggan**: Mengingat preferensi selama sesi dukungan yang panjang
- **Asisten Pribadi**: Melacak proyek yang sedang berjalan tanpa perlu menjelaskan konteks berulang kali
- **Tutor Pendidikan**: Mempertahankan kemajuan siswa di berbagai interaksi

### Langkah Selanjutnya
- Mengimplementasikan alat scratchpad lengkap dengan persistensi berbasis file
- Menambahkan pemangkasan riwayat otomatis setelah ringkasan dibuat
- Menggabungkan dengan basis data vektor untuk pencarian memori semantik
- Membangun agen yang dapat melanjutkan percakapan beberapa hari kemudian dengan konteks penuh


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk memberikan terjemahan yang akurat, harap diingat bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sahih. Untuk informasi yang penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
